# 1 Pandas高级数据处理

In [1]:
import numpy as np
import pandas as pd

## 1.1 级联

汇总全年销售数据

In [2]:
df1 = pd.DataFrame(data=[[1,2,3]], columns=list('ABC'))
df1

,A,B,C
0,1,2,3


In [3]:
df2 = pd.DataFrame(data=[[2,3,4]], columns=list('ABC'))
df2

,A,B,C
0,2,3,4


In [4]:
# 级联语法的核心就是索引对齐
# 级联的应用场景：不同期，但是结构相同的数据汇总
# object对象
# axis=0列索引对齐，axis=1行索引对齐
res1 = pd.concat([df1, df2], axis=0)

In [5]:
res2 = pd.concat([df1, df2], axis=1)

In [6]:
res1.loc[0]

,A,B,C
0,1,2,3
0,2,3,4


In [7]:
res2.loc[:, 'A']

,A,A
0,1,2


In [8]:
# 校验级联之后是否有重复索引
# pd.concat((df1, df2), verify_integrity=True)

In [9]:
# pd.concat((df1, df2), axis=1, verify_integrity=True)

In [10]:
# 通过多层级索引来处理重复索引的问题
pd.concat((df1, df2), axis=1, keys=['上半年', '下半年'], names=['周期', '产品'])

周期 上半年       下半年      
产品   A  B  C   A  B  C
0    1  2  3   2  3  4

In [11]:
# 通过忽略索引的方式来处理重复索引的问题
pd.concat((df1, df2), ignore_index=True)

,A,B,C
0,1,2,3
1,2,3,4


In [12]:
df3 = pd.DataFrame(data=[[1,2,3,4]], columns=list('CDAB'))
df3

,C,D,A,B
0,1,2,3,4


In [13]:
df2

,A,B,C
0,2,3,4


In [14]:
pd.concat((df3, df2), sort=False)

,C,D,A,B
0,1,2.0,3,4
0,4,NaN,2,3


In [15]:
df4=pd.DataFrame(data=np.random.randint(0,100,size=(3,4)),columns=list('ABCD'))
df5=pd.DataFrame(data=np.random.randint(-100,0,size=(4,3)),columns=list('BDE'))

In [16]:
display(df4, df5)

,A,B,C,D
0,5,38,29,46
1,43,24,13,80
2,5,40,69,91


,B,D,E
0,-2,-80,-31
1,-5,-47,-53
2,-50,-79,-78
3,-42,-95,-89


In [17]:
# outer 保留级联方向的所有【标签】并集
# inner 保留级联方向的共有【标签】交集
pd.concat((df4, df5, df1), sort=True, join='outer')

,A,B,C,D,E
0,5.0,38,29.0,46.0,NaN
1,43.0,24,13.0,80.0,NaN
2,5.0,40,69.0,91.0,NaN
0,NaN,-2,NaN,-80.0,-31.0
1,NaN,-5,NaN,-47.0,-53.0
2,NaN,-50,NaN,-79.0,-78.0
3,NaN,-42,NaN,-95.0,-89.0
0,1.0,2,3.0,NaN,NaN


## 1.2 合并

合并就是根据两张表的公共信息，把两张表的数据汇总的方法。

合并以列的内容为参考标准，不存在行合并，都是列合并合并的列通常是离散型数据。

可以是数值型，也可以是类别型数据合并的列之间存在一对一、一对多、多对多关系，否则合并结果为空

### 1.2.1 计算上半年订单总额GMV

参数：left_onright_on

In [23]:
first_half_year=pd.read_excel('合并表格案例.xlsx',sheet_name=0)
second_half_year=pd.read_excel('合并表格案例.xlsx',sheet_name=1)
display(first_half_year.head(), second_half_year.head())

,用户ID,商品ID,订单ID,购买数量
0,lucy,10001,1009,1
1,jack,10002,1002,2
2,lucy,10003,1007,1
3,alex,10004,1010,1
4,mery,10005,1008,1


,用户ID,商品ID,订单ID,购买数量
0,tom,10006,2001,1
1,oldshang,10003,2002,1
2,佩奇,10004,2003,1
3,小明,10004,2004,2
4,小红,10001,2005,1


In [20]:
user_table=pd.read_excel('合并表格案例.xlsx',sheet_name=2)
user_table

,用户ID,地区,VIP等级,手机号
0,lucy,北京,3,13054344433
1,智哥,深圳,3,13046798795
2,mery,北京,2,17877659878
3,jack,上海,4,18635482221
4,alex,北京,1,17601002323
5,tom,深圳,2,18910538799
6,oldshang,北京,3,17699887678
7,佩奇,上海,2,15600140101
8,小明,上海,2,18789897788
9,小红,北京,3,17625745653


In [28]:
product_table = pd.read_excel('合并表格案例.xlsx', sheet_name=3)
product_table.head()

,商品ID,商品类别,商品品牌,商品单价
0,10001,笔记本,华为,8000
1,10002,笔记本,小米,7600
2,10003,鼠标,华为,300
3,10004,鼠标,apple,600
4,10005,键盘,apple,1000


In [26]:
return_table = pd.read_excel('合并表格案例.xlsx', sheet_name=4)
return_table.head()

,订单_id,退货状态
0,1003,退货中
1,1004,退货中
2,1005,退货完成
3,1011,退货完成
4,1014,退货中


In [29]:
#两张表合并时，默认是根据所有的相同字段名称的列来进行合并
pd.merge(left=first_half_year, right=product_table)

,用户ID,商品ID,订单ID,购买数量,商品类别,商品品牌,商品单价
0,lucy,10001,1009,1,笔记本,华为,8000
1,alex,10001,1001,3,笔记本,华为,8000
2,tom,10001,1011,1,笔记本,华为,8000
3,jack,10002,1002,2,笔记本,小米,7600
4,佩奇,10002,1013,1,笔记本,小米,7600
5,mery,10002,1015,2,笔记本,小米,7600
6,lucy,10003,1007,1,鼠标,华为,300
7,alex,10004,1010,1,鼠标,apple,600
8,智哥,10004,1004,1,鼠标,apple,600
9,mery,10005,1008,1,键盘,apple,1000


### 1.2.2 获取上半年用户地区，查看各地区订单数量

参数：how

### 1.2.3 找出上半年和下半年购买过相同商品的用户

参数解释：on，suffixes

一个用户，上半年和下半年都购买了同一个商品